# Resgatécnica — Análise de Aderência com Claude Haiku (Anthropic API)

**Objetivo:** Analisar aderência das licitações ao portfólio da Resgatécnica usando Claude Haiku 4.5.

**Sequência:**
1. Upload do ZIP com os dados
2. Instalação das dependências Python
3. Configuração da chave da API Anthropic
4. Execução do agente
5. Telemetria: custo, latência, qualidade JSON
6. Download dos resultados

> **Não precisa de GPU** — pode usar o runtime padrão (CPU) do Colab.

## Célula 1 — Upload do ZIP

In [ ]:
from google.colab import files
import os

print('Faça upload do arquivo resgatecnica_colab.zip')
uploaded = files.upload()

zip_name = [k for k in uploaded.keys() if k.endswith('.zip')]
if not zip_name:
    raise RuntimeError('Nenhum arquivo .zip encontrado no upload.')
zip_name = zip_name[0]
print(f'Upload recebido: {zip_name} ({len(uploaded[zip_name]):,} bytes)')

## Célula 2 — Extrair ZIP e verificar estrutura

In [ ]:
import zipfile, pathlib

WORK_DIR = pathlib.Path('/content/resgatecnica')
WORK_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(WORK_DIR)

checks = {
    'pncp_agente.py':       WORK_DIR / 'pncp_agente.py',
    'pncp_licitacoes.json': WORK_DIR / 'pncp_licitacoes.json',
    'exclusoes.yaml':       WORK_DIR / 'exclusoes.yaml',
    'prompts/':             WORK_DIR / 'prompts',
    'editais/':             WORK_DIR / 'editais',
}
for nome, caminho in checks.items():
    status = '✓' if caminho.exists() else '✗ FALTANDO'
    print(f'  {status}  {nome}')

n_editais = sum(1 for d in (WORK_DIR / 'editais').iterdir()
                if d.is_dir() and (d / 'itens.json').exists())
print(f'\nLicitações com itens.json: {n_editais}')
print(f'Diretório de trabalho: {WORK_DIR}')

## Célula 3 — Instalar dependências

In [ ]:
!pip install anthropic jsonschema requests pyyaml -q
print('Dependências instaladas.')

## Célula 4 — Configurar API Anthropic

Cole sua chave da API no campo abaixo.  
> ⚠️ **Não salve nem compartilhe este notebook com a chave preenchida.**

In [ ]:
import os

os.chdir(WORK_DIR)
print(f'Diretório atual: {os.getcwd()}')

# -----------------------------------------------------------------------
# Cole sua chave da API Anthropic abaixo.
# -----------------------------------------------------------------------
ANTHROPIC_API_KEY = 'sk-ant-...'   # ← substitua pela sua chave real

MODELO          = 'claude-haiku-4-5'
MAX_PROCESSOS   = '10'   # 0 = sem limite (todas as licitações)

os.environ['PROVEDOR']          = 'anthropic'
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
os.environ['ANTHROPIC_MODELO']  = MODELO
os.environ['MAX_PROCESSOS']     = MAX_PROCESSOS

if ANTHROPIC_API_KEY == 'sk-ant-...' or len(ANTHROPIC_API_KEY) < 20:
    print('⚠ ATENÇÃO: chave não preenchida. Substitua sk-ant-... pela sua chave real.')
else:
    chave_fmt = ANTHROPIC_API_KEY[:12] + '...' + ANTHROPIC_API_KEY[-4:]
    print(f'✓ Configuração:')
    print(f'  PROVEDOR          = anthropic')
    print(f'  ANTHROPIC_MODELO  = {MODELO}')
    print(f'  MAX_PROCESSOS     = {MAX_PROCESSOS}  (0 = todas)')
    print(f'  ANTHROPIC_API_KEY = {chave_fmt}')
    print()
    # Estimativa de custo
    n = int(MAX_PROCESSOS) if MAX_PROCESSOS != '0' else n_editais
    custo_est = n * 2 * 0.026
    print(f'  Custo estimado ({n} licitações × ~2 chamadas × $0,026): ~${custo_est:.2f} USD')

## Célula 5 — Executar agente

In [ ]:
import subprocess, time, os, re

if os.environ.get('ANTHROPIC_API_KEY', '') in ('', 'sk-ant-...'):
    print('ERRO: configure a chave na Célula 4 antes de executar.')
else:
    print('=' * 60)
    print(f'Iniciando análise com {MODELO}...')
    print(f'Licitações a processar: {MAX_PROCESSOS} (0 = todas)')
    print('=' * 60)

    env = dict(os.environ)
    env['PYTHONUNBUFFERED'] = '1'   # força flush imediato de cada print/log

    t_inicio = time.time()
    proc = subprocess.Popen(
        ['python', 'pncp_agente.py'],
        cwd=str(WORK_DIR),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        encoding='utf-8',
        errors='replace',
    )

    # Padrão de progresso: "[12/126]" no início da mensagem de log
    re_prog = re.compile(r'\[(\d+)/(\d+)\]')

    for linha in proc.stdout:
        linha = linha.rstrip()
        m = re_prog.search(linha)
        if m:
            # Linha de progresso — destaca com prefixo visual
            atual, total = m.group(1), m.group(2)
            pct = int(atual) / int(total) * 100
            barra = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
            print(f'\n▶ [{barra}] {pct:5.1f}%  {linha}', flush=True)
        elif '[ERROR]' in linha or '[WARNING]' in linha:
            print(f'  ⚠  {linha}', flush=True)
        else:
            print(f'  {linha}', flush=True)

    proc.wait()
    t_total = time.time() - t_inicio

    print(f'\n{"=" * 60}')
    if proc.returncode == 0:
        print(f'✓ Concluído em {t_total/60:.1f} min.')
    else:
        print(f'✗ Agente encerrou com código {proc.returncode} após {t_total/60:.1f} min.')

## Célula 6 — Telemetria e custo

In [ ]:
import json, statistics
from pathlib import Path

tel_path = WORK_DIR / 'pncp_telemetria.jsonl'
if not tel_path.exists():
    print('pncp_telemetria.jsonl não encontrado — execute a Célula 5 primeiro.')
else:
    linhas = []
    with open(tel_path) as f:
        for linha in f:
            try:
                linhas.append(json.loads(linha))
            except Exception:
                pass

    llm = [l for l in linhas if l.get('provedor') == 'anthropic']
    pf  = [l for l in linhas if l.get('provedor') == 'pre_filtro']

    if not llm:
        print('Nenhuma chamada Anthropic encontrada.')
    else:
        llm_ok  = [l for l in llm if l.get('lat_ms') is not None]
        llm_err = [l for l in llm if l.get('lat_ms') is None]

        lats      = [l['lat_ms']  for l in llm_ok]
        toks_out  = [l['tok_out'] for l in llm_ok if l.get('tok_out')]
        schema_ok = sum(1 for l in llm if l.get('schema_ok'))
        custo     = sum(l.get('custo_usd') or 0 for l in llm)
        cache_r   = sum(l.get('cache_r')   or 0 for l in llm)
        cache_w   = sum(l.get('cache_w')   or 0 for l in llm)

        print(f'{"="*55}')
        print(f'BENCHMARK — {MODELO}')
        print(f'{"="*55}')
        print(f'Chamadas LLM totais:    {len(llm)}')
        print(f'  Com resposta:         {len(llm_ok)}')
        print(f'  Sem resposta (erro):  {len(llm_err)}')
        print(f'Pré-filtradas (sem LLM):{len(pf)}')
        print(f'Schema válido:          {schema_ok}/{len(llm)} ({schema_ok/len(llm)*100:.0f}%)')
        print(f'\nCusto total:            ${custo:.4f} USD')
        print(f'  Cache read tokens:    {cache_r:,}')
        print(f'  Cache write tokens:   {cache_w:,}')
        if len(llm_ok) > 1:
            print(f'  Custo/licitação (est.): ${custo/len(set(l.get("processo") for l in llm_ok)):.4f}')

        if lats:
            print(f'\nLatência (s):')
            print(f'  Mínima:   {min(lats)/1000:.1f}s')
            print(f'  Média:    {statistics.mean(lats)/1000:.1f}s')
            print(f'  Mediana:  {statistics.median(lats)/1000:.1f}s')
            print(f'  Máxima:   {max(lats)/1000:.1f}s')

        if toks_out and lats:
            tok_s = sum(toks_out) / (sum(lats) / 1000)
            print(f'\nTokens de saída:')
            print(f'  Média:    {statistics.mean(toks_out):.0f}')
            print(f'  Máximo:   {max(toks_out)}')
            print(f'  tok/s:    {tok_s:.1f}')

        if lats:
            procs_ok     = set(l.get('processo') for l in llm_ok)
            n_lic        = len(procs_ok)
            n_itens_tot  = sum(l.get('n_itens', 0) for l in llm_ok)
            tempo_s      = sum(lats) / 1000
            lic_h        = (n_lic / tempo_s) * 3600 if tempo_s > 0 else 0
            media_itens  = n_itens_tot / n_lic if n_lic else 0
            print(f'\nThroughput:')
            print(f'  Licitações processadas: {n_lic}')
            print(f'  Média itens/licitação:  {media_itens:.1f}')
            print(f'  Tempo total inferência: {tempo_s/60:.1f} min')
            print(f'  Licitações/hora:        {lic_h:.0f}')

        print(f'\n{"="*55}')
        print('Detalhes por chamada:')
        print(f'{"processo":<42} {"lote":<7} {"n":<4} {"lat_s":<8} {"tok_out":<8} {"custo":<9} {"schema"}')
        print('-' * 90)
        for l in llm:
            proc    = (l.get('processo') or '')[-38:]
            lat_ms  = l.get('lat_ms')
            tok_out = l.get('tok_out')
            lat_s   = f'{lat_ms/1000:.1f}s' if lat_ms is not None else 'N/A'
            custo_l = f'${l.get("custo_usd") or 0:.4f}'
            erro    = f' ← {str(l.get("erro",""))[:35]}' if l.get('erro') else ''
            print(f'{proc:<42} {str(l.get("lote","?")):<7} {l.get("n_itens",0):<4} '
                  f'{lat_s:<8} {str(tok_out or "N/A"):<8} {custo_l:<9} '
                  f'{"OK" if l.get("schema_ok") else "FAIL"}{erro}')

## Célula 7 — Classificações por licitação

In [ ]:
import json
from pathlib import Path

editais_dir = WORK_DIR / 'editais'
resultados  = []

for pasta in sorted(editais_dir.iterdir()):
    adh = pasta / 'aderencia.json'
    if not adh.exists():
        continue
    with open(adh) as f:
        r = json.load(f)
    if r.get('_pre_filtrado'):
        continue

    # Modo full: parecer_geral fica dentro de resumo_licitacao
    # Modo compacto: fica na raiz
    if 'resumo_licitacao' in r:
        resumo  = r['resumo_licitacao']
        parecer = resumo.get('parecer_geral', '?')
        contexto = resumo.get('contexto_dominante', '')[:80]
    else:
        parecer  = r.get('parecer_geral', '?')
        contexto = ''

    rec        = r.get('recomendacao_comercial', {}) or {}
    prioridade = rec.get('nivel_prioridade', '?')
    stats      = r.get('estatisticas', {})

    resultados.append({
        'processo':   pasta.name,
        'parecer':    parecer,
        'prioridade': prioridade,
        'contexto':   contexto,
        'AD':   stats.get('aderencia_direta', 0),
        'PF':   stats.get('aderencia_parcial_forte', 0),
        'Pw':   stats.get('aderencia_parcial_fraca', 0),
        'FPL':  stats.get('falso_positivo_lexical', 0),
        'NA':   stats.get('nao_aderente', 0),
        'itens': len(r.get('itens_analisados', [])),
    })

if not resultados:
    print('Nenhum resultado. Execute a Célula 5 primeiro.')
else:
    print(f'Licitações analisadas: {len(resultados)}\n')
    print(f'{"processo":<45} {"parecer":<22} {"prior":<10} {"AD":<4} {"PF":<4} {"Pw":<4} {"FPL":<5} {"NA":<4} {"itens"}')
    print('-' * 105)
    for r in resultados:
        print(f'{r["processo"][-42:]:<45} {r["parecer"]:<22} {r["prioridade"]:<10} '
              f'{r["AD"]:<4} {r["PF"]:<4} {r["Pw"]:<4} {r["FPL"]:<5} {r["NA"]:<4} {r["itens"]}')
        if r["contexto"]:
            print(f'  └ {r["contexto"]}')

    from collections import Counter
    pareceres   = Counter(r['parecer']    for r in resultados)
    prioridades = Counter(r['prioridade'] for r in resultados)
    print(f'\nPareceres:   {dict(pareceres)}')
    print(f'Prioridades: {dict(prioridades)}')

## Célula 8 — Download dos resultados

Gera `resultados_colab_haiku.zip` com telemetria, sumário e todos os `aderencia.json`.

In [ ]:
from google.colab import files as colab_files
import zipfile

saida_zip = WORK_DIR / 'resultados_colab_haiku.zip'

with zipfile.ZipFile(saida_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for nome in ['pncp_aderencias.json', 'pncp_telemetria.jsonl', 'pncp_itens_consolidado.json']:
        p = WORK_DIR / nome
        if p.exists():
            zf.write(p, nome)
            print(f'  + {nome}  ({p.stat().st_size / 1024:.0f} KB)')

    n_adh = 0
    for pasta in (WORK_DIR / 'editais').iterdir():
        adh = pasta / 'aderencia.json'
        if adh.exists():
            zf.write(adh, f'editais/{pasta.name}/aderencia.json')
            n_adh += 1
    print(f'  + editais/*/aderencia.json  ({n_adh} arquivos)')

print(f'\nFazendo download de {saida_zip.name}...')
colab_files.download(str(saida_zip))